## FABlib API References Examples

- [fablib.show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config)
- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [slice.list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodesß)
- [slice.show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete)

## Main purpose

- Configure DYNAMOS
- Acquire SSH commands to connect to different nodes
- Install and Uninstall DYNAMOS manually

In [1]:
import datetime
import json
import asyncio
from configuration import SLICE_NAME
from utils import upload_and_execute_file, override_configuration_files, upload_file, execute_file

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
Bastion Host,bastion.fabric-testbed.net
Bastion Username,apipilikas_0000444352
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


## Setting up variables

In [2]:
%%time
image = "default_ubuntu_24"

# Please adhere to the following regex for naming: /[a-z][a-z0-9]+/
# note: see above, renamed the agent names to only have hyphens, not underscores 

node_configurations = [
    {
        "type": "control",
        "cores": 2,
        "ram": 8,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "dynamos",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "server",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientone",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "LOSA",
        "host": "losa-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clienttwo",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientthree",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "thirdparty",
        "name": "surf",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    }
]

sites = list(set([configuration["site"] for configuration in node_configurations]))
agents = [configuration["name"] for configuration in node_configurations if configuration["type"] == "agent"]
thirdparties = [configuration["name"] for configuration in node_configurations if configuration["type"] == "thirdparty"]

def create_node(slice, configuration):
    if (configuration["type"] == "control"): 
        configuration["name"] = "control"

    if (configuration["type"] == "dynamos"): 
        configuration["name"] = "dynamos"
    
    return slice.add_node(name=configuration["name"], 
                          site=configuration["site"], 
                          host=configuration["host"], 
                          cores=configuration["cores"], 
                          ram=configuration["ram"], 
                          disk=configuration["disk"], 
                          validate=True, 
                          raise_exception=True, 
                          image=image)
    

CPU times: user 23 μs, sys: 0 ns, total: 23 μs
Wall time: 28.6 μs


In [3]:
%%time
slice = fablib.get_slice(name=SLICE_NAME);
nodes = slice.get_nodes();

User: apipilikas@gmail.com bastion key is valid!
Configuration is valid
CPU times: user 390 ms, sys: 22.3 ms, total: 412 ms
Wall time: 5 s


## Print SSH information

In [4]:
# Print ssh information
try:
    # Get slice nodes
    for node in slice.get_nodes():
        print("---------------------------------------------------------------------------")
        print(f"> Node: {node.get_name()}")
        # Get the original SSH command
        original_ssh_command = node.get_ssh_command()
        # Print SSH commands to get into the nodes
        print(f"-- SSH Command from FABRIC:\n   {original_ssh_command}")
        # Replace the file paths in the SSH command
        updated_ssh_command = original_ssh_command.replace(
            "/home/fabric/work/fabric_config/slice_key", "C:/Users/apipi/.ssh/slice_key"
        ).replace(
            "/home/fabric/work/fabric_config/ssh_config", "C:/Users/apipi/.ssh/fabric_ssh_config"
        )
        # Print the updated SSH command
        print(f"-- SSH Command locally:\n   {updated_ssh_command}")

        # Print SSH command forwarding
        token = "%FORWARD%"
        
        ssh_command_parts = updated_ssh_command.split()
        ssh_command_parts.insert(-1, "-L")
        ssh_command_parts.insert(-1, token)

        forward_ssh_command = " ".join(ssh_command_parts)
        api_forward_port = "8080:localhost:8080"
        api_gateway_ssh_command = forward_ssh_command.replace(token, api_forward_port)
        print(f"-- SSH Command forward port api-gateway:\n   {api_gateway_ssh_command}")

        grafana_forward_port = "3000:localhost:3000"
        grafana_ssh_command = forward_ssh_command.replace(token, grafana_forward_port)
        print(f"-- SSH Command forward port grafana:\n   {grafana_ssh_command}")

        prometheus_forward_port = "9090:localhost:9090"
        prometheus_ssh_command = forward_ssh_command.replace(token, prometheus_forward_port)
        print(f"-- SSH Command forward port prometheus:\n   {prometheus_ssh_command}")

        orchestrator_port = "8090:localhost:8090"
        orchestrator_ssh_command = forward_ssh_command.replace(token, orchestrator_port)
        print(f"-- SSH Command forward port orchestrator:\n   {orchestrator_ssh_command}")

        jaeger_forward_port = "16686:localhost:16686"
        prometheus_ssh_command = forward_ssh_command.replace(token, jaeger_forward_port)
        print(f"-- SSH Command forward port jaeger:\n   {prometheus_ssh_command}")

        all_forward_port = f"{api_forward_port} -L {orchestrator_port} -L {grafana_forward_port} -L {prometheus_forward_port} -L {jaeger_forward_port}"
        all_ssh_command = forward_ssh_command.replace(token, all_forward_port)
        print(f"-- SSH Command forward port ALL:\n   {all_ssh_command}")
    
except Exception as e:
    print(f"Fail: {e}")
    traceback.print_exc()

---------------------------------------------------------------------------
> Node: control
-- SSH Command from FABRIC:
   ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command locally:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port api-gateway:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 8080:localhost:8080 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port grafana:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 3000:localhost:3000 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port prometheus:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 9090:localhost:9090 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward

In [5]:
%%time
def get_ip(node):
    interface = node.get_interface(network_name=f"Network-{node.get_site()}")
    return interface.get_ip_addr()

nodes_dict= dict()

for node in nodes[:]:
    ip = get_ip(node)
    name = node.get_name()
    nodes_dict[name] = {"ip": ip, "node": node}
    print(f"{name}: {ip}")

print(nodes_dict)


control: 10.145.1.2
dynamos: 10.145.1.3
server: 10.145.1.4
aggregator: 10.145.1.5
authority: 10.145.1.6
clientone: 10.145.1.7
clienttwo: 10.145.1.8
clientthree: 10.145.1.9
surf: 10.145.1.10
{'control': {'ip': '10.145.1.2', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df502134d70>}, 'dynamos': {'ip': '10.145.1.3', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213e490>}, 'server': {'ip': '10.145.1.4', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213e0d0>}, 'aggregator': {'ip': '10.145.1.5', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213e210>}, 'authority': {'ip': '10.145.1.6', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213e350>}, 'clientone': {'ip': '10.145.1.7', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213df90>}, 'clienttwo': {'ip': '10.145.1.8', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7df50213e5d0>}, 'clientthree': {'ip': '1

## Setting up main node

In [6]:
main_node=nodes_dict['control']['node']
print(f"The main node for now on is [{main_node.get_name()}]")

The main node for now on is [control]


## Setting up directory and overriding basic files

In [7]:
upload_and_execute_file(main_node, local_file_path="node_scripts/define_etcd_data.sh", remote_file_path="define_etcd_data.sh")

Uploading file from local path [node_scripts/define_etcd_data.sh] to remote path [define_etcd_data.sh] ...
Executing file [define_etcd_data.sh] ...
Cleaning scattered-directive-energy-monitoring folder...
Cloning  branch fed-encrypt-dynamic
Cloning into 'scattered-directive-energy-monitoring'...


In [8]:
# Optionally override the installation scripts (highly suggested as these scripts contain the nodeName that you deploy dynamos)
override_configuration_files(main_node)
main_node.execute("find /home/ubuntu/scattered-directive-energy-monitoring -type f -name '*.sh' -exec sed -i 's/\\r$//' {} + -exec chmod +x {} +")

Uploading file from local path [dynamos.conf] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/dynamos.conf] ...
Uploading file from local path [overriden_files/temp-pod.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/configuration/temp-pod.yaml] ...
Uploading file from local path [overriden_files/core-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/core/values.yaml] ...
Uploading file from local path [overriden_files/orchestrator-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/orchestrator/values.yaml] ...
Uploading file from local path [overriden_files/namespaces-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/namespaces/values.yaml] ...
Uploading file from local path [overriden_files/api-gateway-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/api-gateway/value

('', '')

## Install DYNAMOS

In [11]:
%%time
main_node.execute("""
sed -i 's/\\r$//' ~/scattered-directive-energy-monitoring/configuration/dynamos-configuration.sh
~/scattered-directive-energy-monitoring/configuration/dynamos-configuration.sh fabric
""")


=============== Started setting up DYNAMOS (fabric) ===============

Setting up paths...

Agents discovered: aggregator,authority,server,clientone,clienttwo,clientthree

Generating agents and third parties charts...

Adding agents...
- agent 'aggregator'
- agent 'authority'
- agent 'server'
- agent 'clientone'
- agent 'clienttwo'
- agent 'clientthree'

Adding third parties...
definitions_example.json copied over definitions.json to ensure a clean file
Generating RabbitMQ password...

Replacing tokens...

Installing namespaces...

Release "namespaces" does not exist. Installing it now.
NAME: namespaces
LAST DEPLOYED: Fri Jul 10 10:29:33 2026
NAMESPACE: default
STATUS: deployed
REVISION: 1
TEST SUITE: None

Preparing PVC...


Applying temp-pod.yaml (Attempt 1/3)...
pod/temp-pod created
pod/temp-pod-orch created

Waiting for temp-pod to be Running...

pod/temp-pod condition met
pod/temp-pod-orch condition met
./
./datasets.json
./agreements.json
./requestType.json
./archetype.json
./micr

('\n=============== Started setting up DYNAMOS (fabric) ===============\n\nSetting up paths...\n\nAgents discovered: aggregator,authority,server,clientone,clienttwo,clientthree\n\nGenerating agents and third parties charts...\n\nAdding agents...\n- agent \'aggregator\'\n- agent \'authority\'\n- agent \'server\'\n- agent \'clientone\'\n- agent \'clienttwo\'\n- agent \'clientthree\'\n\nAdding third parties...\ndefinitions_example.json copied over definitions.json to ensure a clean file\nGenerating RabbitMQ password...\n\nReplacing tokens...\n\nInstalling namespaces...\n\nRelease "namespaces" does not exist. Installing it now.\nNAME: namespaces\nLAST DEPLOYED: Fri Jul 10 10:29:33 2026\nNAMESPACE: default\nSTATUS: deployed\nREVISION: 1\nTEST SUITE: None\n\nPreparing PVC...\n\n\nApplying temp-pod.yaml (Attempt 1/3)...\npod/temp-pod created\npod/temp-pod-orch created\n\nWaiting for temp-pod to be Running...\n\npod/temp-pod condition met\npod/temp-pod-orch condition met\n./\n./datasets.json\n

## Uninstall DYNAMOS

In [10]:
cleanup_command = """
helm uninstall surf -n default --ignore-not-found
kubectl patch pv etcd-pv -p '{"metadata":{"finalizers":null}}'
kubectl delete pvc --all -n core --wait=false
kubectl delete pvc --all -n orchestrator --wait=false
"""
main_node.execute(cleanup_command)
main_node.execute("""
sed -i 's/\\r$//' /home/ubuntu/scattered-directive-energy-monitoring/configuration/uninstall-dynamos.sh
/home/ubuntu/scattered-directive-energy-monitoring/configuration/uninstall-dynamos.sh
""")

# command = "helm uninstall agents api-gateway core orchestrator namespaces prometheus thirdparties"
# nodes_dict['control']['node'].execute(command)

release "surf" uninstalled
persistentvolume/etcd-pv patched
persistentvolumeclaim "etcd-data-etcd-0" deleted
persistentvolumeclaim "etcd-data-etcd-1" deleted
persistentvolumeclaim "etcd-data-etcd-2" deleted
persistentvolumeclaim "rabbit-pvc" deleted
persistentvolumeclaim "rabbitmq-data-pvc" deleted
persistentvolumeclaim "rabbitmq-log-pvc" deleted
persistentvolumeclaim "etcd-pvc" deleted

=============== Started uninstalling DYNAMOS ===============

Uninstalling DYNAMOS namespaces...

release "nginx" uninstalled
These resources were kept due to the resource policy:
[Namespace] core
[Namespace] orchestrator
[Namespace] clientthree
[Namespace] aggregator
[Namespace] authority
[Namespace] uva
[Namespace] vu
[Namespace] surf
[Namespace] ingress
[Namespace] api-gateway
[Namespace] server
[Namespace] clientone
[Namespace] clienttwo

release "namespaces" uninstalled
release "core" uninstalled
release "orchestrator" uninstalled
release "agents" uninstalled
release "thirdparties" uninstalled
rel

('\n=============== Started uninstalling DYNAMOS ===============\n\nUninstalling DYNAMOS namespaces...\n\nrelease "nginx" uninstalled\nThese resources were kept due to the resource policy:\n[Namespace] core\n[Namespace] orchestrator\n[Namespace] clientthree\n[Namespace] aggregator\n[Namespace] authority\n[Namespace] uva\n[Namespace] vu\n[Namespace] surf\n[Namespace] ingress\n[Namespace] api-gateway\n[Namespace] server\n[Namespace] clientone\n[Namespace] clienttwo\n\nrelease "namespaces" uninstalled\nrelease "core" uninstalled\nrelease "orchestrator" uninstalled\nrelease "agents" uninstalled\nrelease "thirdparties" uninstalled\nrelease "api-gateway" uninstalled\nrelease "surf" uninstalled\n\nUninstalling monitoring namespaces...\n\nrelease "prometheus" uninstalled\nrelease "kepler" uninstalled\nrelease "monitoring" uninstalled\n\nUninstalling nginx...\n\nrelease "nginx" uninstalled\nrelease "nginx" uninstalled\n\nClearing old jobs...\n\n- Clearing pods for agent: [\'aggregator\']\njob.b

In [9]:
main_node.execute("""
./scattered-directive-energy-monitoring/scripts/forward_ports.sh
""")


=============== Waiting for pods to be ready ===============

Waiting for service prometheus-kube-prometheus-prometheus in namespace monitoring to be ready...
> Service prometheus-kube-prometheus-prometheus is ready!
Waiting for service api-gateway in namespace api-gateway to be ready...
> Service api-gateway is ready!
Waiting for service orchestrator in namespace orchestrator to be ready...
> Service orchestrator is ready!
Waiting for service prometheus-grafana in namespace monitoring to be ready...
> Service prometheus-grafana is ready!

=============== Started port forwarding ===============

Forwarding from 127.0.0.1:8080 -> 8080
Forwarding from [::1]:8080 -> 8080
Forwarding from 127.0.0.1:9090 -> 9090
Forwarding from [::1]:9090 -> 9090
Forwarding from 127.0.0.1:8090 -> 8080
Forwarding from [::1]:8090 -> 8080
Forwarding from 127.0.0.1:3000 -> 3000
Forwarding from [::1]:3000 -> 3000
Forwarding from 127.0.0.1:16686 -> 16686
Forwarding from [::1]:16686 -> 16686

All ports forwarded! 

KeyboardInterrupt: 

In [10]:
main_node.execute("pkill -f \"port-forward\"")

('', '')

In [12]:
upload_file(main_node, "overriden_files/temp/requestType.json", "/home/ubuntu/scattered-directive-energy-monitoring/configuration/etcd_launch_files/requestType.json")

Uploading file from local path [overriden_files/temp/requestType.json] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/configuration/etcd_launch_files/requestType.json] ...


In [20]:
for node in nodes[:]:
    print(f"> Rebooting node: {node.get_name()}") 
    node.os_reboot()

> Rebooting node: control
> Rebooting node: dynamos
> Rebooting node: server
> Rebooting node: clientone
> Rebooting node: clienttwo
> Rebooting node: clientthree
> Rebooting node: surf


In [10]:
main_node.execute("sudo apt-get update && sudo apt-get install -y docker.io apt-transport-https curl python3 python3-venv python3-pip ca-certificates gpg")
main_node.execute("sudo systemctl restart docker")
main_node.execute("sudo chmod 666 /var/run/docker.sock")

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:5 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:7 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1704 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/main Translation-en [268 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Components [42.4 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1191 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe Translation-en [230 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Components [74.3 kB]
Get:13 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2007 kB

('', '')

In [77]:
for node in nodes[:]:
    print(f"> Cleaning cache: {node.get_name()}") 
    # node.execute("sudo crictl rmi --prune")
    node.execute("""
    sudo crictl rmi docker.io/apipilikas/sidecar:latest
    """)

> Cleaning cache: control
time="2026-06-06T19:21:54Z" level=error msg="no such image docker.io/apipilikas/sidecar:latest"
time="2026-06-06T19:21:54Z" level=fatal msg="unable to remove the image(s)"
> Cleaning cache: dynamos
time="2026-06-06T19:21:57Z" level=error msg="no such image docker.io/apipilikas/sidecar:latest"
time="2026-06-06T19:21:57Z" level=fatal msg="unable to remove the image(s)"
> Cleaning cache: server
time="2026-06-06T19:21:59Z" level=error msg="no such image docker.io/apipilikas/sidecar:latest"
time="2026-06-06T19:21:59Z" level=fatal msg="unable to remove the image(s)"
> Cleaning cache: aggregator
time="2026-06-06T19:22:01Z" level=error msg="no such image docker.io/apipilikas/sidecar:latest"
time="2026-06-06T19:22:01Z" level=fatal msg="unable to remove the image(s)"
> Cleaning cache: authority
time="2026-06-06T19:22:03Z" level=error msg="no such image docker.io/apipilikas/sidecar:latest"
time="2026-06-06T19:22:03Z" level=fatal msg="unable to remove the image(s)"
> Clea